In [27]:
import os

os.listdir("/kaggle/input")

['datasets']

In [28]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/lorenzoarcioni
/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data
/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data/labels
/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data/images
/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data/labels-YOLO


In [29]:
import shutil

SOURCE = "/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data"
DEST = "/kaggle/working/road_damage"

if os.path.exists(DEST):
    shutil.rmtree(DEST)

shutil.copytree(SOURCE, DEST)

print("Dataset Copied Successfully")

Dataset Copied Successfully


In [30]:
import os

for root, dirs, files in os.walk("/kaggle/working/road_damage"):
    print(root)

/kaggle/working/road_damage
/kaggle/working/road_damage/labels-YOLO
/kaggle/working/road_damage/labels
/kaggle/working/road_damage/images


In [31]:
import random
from pathlib import Path

random.seed(42)

base_path = Path("/kaggle/working/road_damage")
images_path = base_path / "images"
labels_path = base_path / "labels-YOLO"

output_path = Path("/kaggle/working/road_damage_dataset")


for split in ["train", "valid", "test"]:
    (output_path / split / "images").mkdir(parents=True, exist_ok=True)
    (output_path / split / "labels").mkdir(parents=True, exist_ok=True)


image_files = list(images_path.glob("*.jpg"))
image_files += list(images_path.glob("*.png"))
image_files += list(images_path.glob("*.jpeg"))

print(f"Total Images: {len(image_files)}")


random.shuffle(image_files)


train_size = int(0.8 * len(image_files))
valid_size = int(0.1 * len(image_files))

train_files = image_files[:train_size]
valid_files = image_files[train_size:train_size + valid_size]
test_files = image_files[train_size + valid_size:]

def copy_data(files, split):
    for img in files:
        shutil.copy(img, output_path / split / "images" / img.name)

        label = labels_path / f"{img.stem}.txt"
        if label.exists():
            shutil.copy(label, output_path / split / "labels" / label.name)

copy_data(train_files, "train")
copy_data(valid_files, "valid")
copy_data(test_files, "test")

print(f"Train Images : {len(train_files)}")
print(f"Valid Images : {len(valid_files)}")
print(f"Test Images  : {len(test_files)}")

Total Images: 2009
Train Images : 1607
Valid Images : 200
Test Images  : 202


In [32]:
dataset = Path("/kaggle/working/road_damage_dataset")

for split in ["train", "valid", "test"]:
    imgs = len(list((dataset / split / "images").glob("*.*")))
    labels = len(list((dataset / split / "labels").glob("*.txt")))

    print(f"{split}:")
    print(f"  Images : {imgs}")

train:
  Images : 1607
valid:
  Images : 200
test:
  Images : 202


In [33]:
import yaml

data = {
    "path": "/kaggle/working/road_damage_dataset",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        0: "Pothole",
        1: "Crack",
        2: "Manhole"
    }
}

with open("/kaggle/working/road_damage_dataset/data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=False)


In [34]:
with open("/kaggle/working/road_damage_dataset/data.yaml") as f:
    print(f.read())

path: /kaggle/working/road_damage_dataset
train: train/images
val: valid/images
test: test/images
names:
  0: Pothole
  1: Crack
  2: Manhole



In [35]:
from ultralytics import YOLO

In [36]:
model = YOLO("yolo11n.pt")

results = model.train(
    data="/kaggle/working/road_damage_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    workers=2,
    project="/kaggle/working/RoadDamage",
    name="YOLO11n"
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/road_damage_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11n-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=

In [37]:
metrics = model.val()

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2623.8±561.4 MB/s, size: 98.4 KB)
val: Scanning /kaggle/working/road_damage_dataset/valid/labels.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 83.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 5.7it/s 2.3s0.2s
                   all        200        484      0.596      0.479      0.497       0.22
               Pothole         92        145      0.546      0.379      0.414      0.162
                 Crack        128        243      0.479      0.288      0.294      0.118
               Manhole         86         96      0.763      0.771      0.783      0.382
Speed: 1.4ms preprocess, 3.4ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /kaggle/worki

In [38]:
model_s = YOLO("yolo11s.pt")

results = model_s.train(
    data="/kaggle/working/road_damage_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    workers=2,
    project="/kaggle/working/RoadDamage",
    name="YOLO11s"
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/road_damage_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11s-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=

In [39]:
metrics = model_s.val()

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2418.1±855.4 MB/s, size: 98.4 KB)
val: Scanning /kaggle/working/road_damage_dataset/valid/labels.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 93.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 4.7it/s 2.7s0.1s
                   all        200        484      0.585      0.507      0.543      0.238
               Pothole         92        145      0.515      0.352      0.391      0.154
                 Crack        128        243      0.453      0.346      0.361      0.141
               Manhole         86         96      0.787      0.823      0.876      0.419
Speed: 1.3ms preprocess, 5.1ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /kaggle/work

In [40]:
model_m = YOLO("yolo11m.pt")

results = model_m.train(
    data="/kaggle/working/road_damage_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    workers=2,
    project="/kaggle/working/RoadDamage",
    name="YOLO11m"
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/road_damage_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11m-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=

In [41]:
metrics = model_m.val()

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 126 layers, 20,032,345 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2294.3±668.7 MB/s, size: 98.4 KB)
val: Scanning /kaggle/working/road_damage_dataset/valid/labels.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 76.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.1it/s 4.2s0.3s
                   all        200        484      0.536      0.574      0.533      0.228
               Pothole         92        145      0.506      0.421       0.43      0.163
                 Crack        128        243      0.391      0.416      0.314      0.114
               Manhole         86         96       0.71      0.885      0.854      0.409
Speed: 1.4ms preprocess, 15.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /kaggle/wo

In [ ]:

model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/road_damage_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0 = 0.0005,
    patience=30,
    workers=2,
    project="/kaggle/working/Hyperparameter",
    name="YOLO11s_E100"
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/road_damage_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11s_E100-3, nbs=64, nms=False, opset=None, optimize=False, opti